## 13.01 词嵌入（word2vec）


### 环境配置


In [1]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import torch
    from torch import nn
    import torch_npu

warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor")
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

import math
from src.utils import Timer


### 练习 13.1.1

**题目：** 计算每个梯度的计算复杂度是多少？如果词表很大，会有什么问题呢？

**解答：** 以跳元模型为例，softmax 形式的条件概率为

$$P(w_o \mid w_c) = \frac{\exp(\mathbf{u}_o^\top \mathbf{v}_c)}{\sum_{j \in \mathcal{V}} \exp(\mathbf{u}_j^\top \mathbf{v}_c)},$$

计算单个中心词向量 $\mathbf{v}_c$ 的梯度需要遍历词表 $\mathcal{V}$ 中所有词的条件概率（求和项），因此梯度计算复杂度约为 $O(|\mathcal{V}|^2)$（每对词都要算一次相似度，词表平方量级）。如果词表很大，会带来两个问题：

- **计算量大**：每个梯度都要对全词表求和，训练一步的时间随词表规模平方增长，难以接受；
- **存储开销大**：两个嵌入矩阵（中心词 + 上下文词）各 $|\mathcal{V}| \times d$，词表越大占用的显存/内存越多，且稀疏梯度更新也慢。

这也是 13.2 节引入负采样与层序 softmax 等近似训练方法的动机。


### 练习 13.1.2

**题目：** 英语中的一些固定短语由多个单词组成，例如“new york”。如何训练它们的词向量？提示：查看 word2vec 论文的第四节。

**解答：** word2vec 论文第四节给出了两种训练固定短语词向量的方法：

1. **用连字符连接成一个词**：例如将 “new york” 写成 “new-york”，使算法把整个短语视为一个词元来训练。这种方法简单直接，但短语长度差异大时容易产生过长的词元，且会破坏单词本身的表示；
2. **用特殊标记替换整个短语**：例如用 `<new_york>` 这样的特殊词元表示短语，在分词阶段就把固定短语替换为单一词元，然后正常训练。

实践中，若语料中固定短语数量少、长度一致，方法 1 更简单；若短语种类多、长度差异大，方法 2 更通用（这也是后续 BPE 等子词方法的雏形，见 13.6 节）。


### 练习 13.1.3

**题目：** 让我们以跳元模型为例来思考 word2vec 设计。跳元模型中两个词向量的点积与余弦相似度之间有什么关系？对于语义相似的一对词，为什么它们的词向量（由跳元模型训练）的余弦相似度可能很高？

**解答：** 余弦相似度的定义为

$$\cos(\theta) = \frac{\mathbf{x}^\top \mathbf{y}}{\|\mathbf{x}\| \|\mathbf{y}\|},$$

分子正是两个向量的点积，分母是二者范数的乘积。因此**点积越大，余弦相似度越高**（两者同向时取最大值 1）。

跳元模型训练时，对语料中的每一对（中心词，上下文词）都要求 $\sigma(\mathbf{u}_o^\top \mathbf{v}_c)$ 尽量大，即**共现词对的向量点积尽量大**。语义相似的词（如 “beautiful” 与 “pretty”）在语料中往往有相似的上下文分布（都常与 “girl”、“view” 等共现），训练过程会把它们推向与这些上下文词方向一致的表示，因此它们的词向量在方向上也趋于一致，余弦相似度自然较高。


---

## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)
